# StyleBot - Entrega 2: guardado, carga e inferencia

**Responsable:** Luis Angel Mendoza Gaytán (simulación, refuerzo e integración).

En esta entrega no se desarrolla aprendizaje por refuerzo. El objetivo es comprobar que los modelos CNN se pueden **guardar, cargar y usar** después en la simulación del robot, y medir cuánto pesan y cuánto tardan en predecir.

| Fecha | Tarea | Sección |
|---|---|---|
| Jueves 12 | Funciones para guardar y cargar `.keras`, predecir una imagen de 28 x 28, medir tiempo y tamaño | 3 y 4 |
| Viernes 13 | Guardar y volver a cargar la Arquitectura A; predicciones con imágenes de Fashion-MNIST | 5 |
| Sábado 14 | Abrir A, B y C, medir tamaño y tiempo de inferencia, elegir la más práctica para el robot | 6 |
| Domingo 15 | Tamaño y tiempo de MobileNetV2 contra la mejor CNN propia | 7 |
| Lunes 16 | Abrir `stylebot_cnn_final.keras` en una sesión limpia, predecir y definir el formato para la Entrega 3 | 8 |

**Reglas:** se usa la misma división del equipo (semilla 42). Las predicciones y métricas de este notebook usan **solo validación**; el conjunto de prueba no se carga, porque el equipo lo reserva para la evaluación final.

**Cómo ejecutarlo:** abrir en Google Colab y usar *Entorno de ejecución > Ejecutar todas*. No necesita GPU: las mediciones se hacen en CPU a propósito. El notebook descarga el repositorio para obtener los modelos.

## 1 Preparación

Todas las mediciones de tiempo se hacen en CPU. Así se parecen más a la computadora que ejecutará la simulación del robot, y los modelos se comparan en las mismas condiciones aunque Colab asigne una GPU.

In [ ]:
import tensorflow as tf

try:
    tf.config.set_visible_devices([], "GPU")   # medir siempre en CPU
except RuntimeError:
    print("Aviso: TensorFlow ya estaba iniciado; reinicia la sesión para medir solo en CPU.")

import hashlib
import json
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

SEMILLA = 42
tf.keras.utils.set_random_seed(SEMILLA)
URL_REPOSITORIO = "https://github.com/krysto9872/StyleBot.git"


def encontrar_repositorio():
    """Usa el repositorio si el notebook se ejecuta dentro de él; si no, lo descarga."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "modelos" / "stylebot_cnn_final.keras").exists():
            return base
    destino = Path("/content/StyleBot") if Path("/content").exists() else Path.cwd() / "StyleBot"
    if not (destino / "modelos" / "stylebot_cnn_final.keras").exists():
        subprocess.run(["git", "clone", "--depth", "1", URL_REPOSITORIO, str(destino)], check=True)
    return destino


RAIZ = encontrar_repositorio()
CARPETA_MODELOS = RAIZ / "modelos"
SALIDA = Path("salidas_03_inferencia")
(SALIDA / "modelos").mkdir(parents=True, exist_ok=True)
(SALIDA / "resultados").mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Dispositivos usados:", [d.device_type for d in tf.config.get_visible_devices()])
print("Repositorio:", RAIZ)
print("Modelos en el repositorio:", sorted(p.name for p in CARPETA_MODELOS.glob("*.keras")))

## 2 Datos de validación

Misma carga y división que `00_datos_cnn.ipynb`. Se comprueba con las huellas del protocolo del líder que los índices son exactamente los del equipo. Se guardan dos versiones de validación: las imágenes originales `uint8` de 0 a 255, como podría recibirlas el robot, y la versión normalizada `float32` de 0 a 1 con forma `(28, 28, 1)`.

In [ ]:
(X_base, y_base), _ = tf.keras.datasets.fashion_mnist.load_data()   # la parte de prueba no se usa

indices = np.arange(len(y_base), dtype=np.int64)
idx_train, idx_val = train_test_split(indices, test_size=0.20, random_state=SEMILLA, stratify=y_base)

X_val_uint8 = X_base[idx_val]
X_val = (X_val_uint8.astype("float32") / 255.0)[..., np.newaxis]
y_val = y_base[idx_val].astype("int64")

protocolo = json.loads((RAIZ / "datos" / "protocolo_datos.json").read_text(encoding="utf-8"))
for nombre, idx in [("idx_train", idx_train), ("idx_val", idx_val)]:
    huella = hashlib.sha256(idx.astype("<i8").tobytes()).hexdigest()
    coincide = huella == protocolo["sha256_indices"][nombre]
    print(f"{nombre}: {'coincide con el protocolo del equipo' if coincide else 'NO coincide'}")
    assert coincide, "La división no es la del equipo"

print("Validación:", X_val.shape, X_val.dtype, "de", float(X_val.min()), "a", float(X_val.max()))
print("Imágenes por categoría:", np.bincount(y_val, minlength=10).tolist())

## 3 Funciones de guardado, carga e inferencia (jueves)

La siguiente celda crea el archivo `inferencia_stylebot.py`, que también está en `src/` del repositorio. Así las mismas funciones sirven en este notebook, en la prueba de sesión limpia de la sección 8 y en la simulación de la Entrega 3.

| Función | Qué hace |
|---|---|
| `guardar_modelo(modelo, ruta)` | Guarda el modelo completo en `.keras` |
| `cargar_modelo(ruta)` | Carga un `.keras` y avisa si el archivo no existe |
| `copia_para_inferencia(modelo, ruta)` | Guarda arquitectura y pesos sin el estado del optimizador (archivo más ligero, mismas predicciones) |
| `predecir_prenda(modelo, imagen)` | Recibe **una** imagen de 28 x 28 y devuelve categoría, nombre, probabilidad, las 3 opciones más probables y el tiempo |
| `predecir_lote(modelo, imagenes)` | Predice muchas imágenes a la vez |
| `medir_tiempo_prediccion(modelo, imagenes)` | Tiempo promedio por imagen, individual y en lote |
| `tamano_modelo_mb(ruta)` | Tamaño del archivo guardado |

In [ ]:
%%writefile inferencia_stylebot.py
"""Inferencia de StyleBot: guardar, cargar y usar modelos .keras.

Responsable: Luis Angel Mendosa Gaytán (Entrega 2, inferencia e integración).
Uso desde la terminal, en una sesión limpia:
    python inferencia_stylebot.py --modelo modelos/stylebot_cnn_final.keras --n 5

Formato para la Entrega 3 (simulación del robot):
    Entrada: una imagen de 28 x 28 en escala de grises, prenda clara sobre fondo oscuro,
             como arreglo de NumPy (28, 28) o (28, 28, 1), uint8 de 0 a 255 o float de 0 a 1.
    Salida:  diccionario con categoria (0-9), nombre, probabilidad, top (3 opciones) y tiempo_ms.
"""
import argparse
import json
import os
import time

import numpy as np
import tensorflow as tf

CATEGORIAS = [
    "Camiseta/top", "Pantalón", "Suéter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Tenis", "Bolso", "Botín",
]


def guardar_modelo(modelo, ruta):
    """Guarda el modelo completo en formato .keras y devuelve la ruta."""
    ruta = os.fspath(ruta)
    if not ruta.endswith(".keras"):
        raise ValueError("La ruta debe terminar en .keras")
    os.makedirs(os.path.dirname(ruta) or ".", exist_ok=True)
    modelo.save(ruta)
    return ruta


def cargar_modelo(ruta):
    """Carga un modelo .keras guardado con guardar_modelo o con modelo.save()."""
    ruta = os.fspath(ruta)
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe el modelo: {ruta}")
    return tf.keras.models.load_model(ruta)


def copia_para_inferencia(modelo, ruta):
    """Guarda solo arquitectura y pesos, sin el estado del optimizador.

    Las predicciones son idénticas, pero el archivo pesa menos porque no guarda
    los momentos de Adam, que solo sirven para seguir entrenando.
    """
    copia = tf.keras.models.clone_model(modelo)
    copia.set_weights(modelo.get_weights())
    return guardar_modelo(copia, ruta)


def tamano_modelo_mb(ruta):
    """Tamaño del archivo guardado, en megabytes."""
    return os.path.getsize(ruta) / (1024 ** 2)


def preparar_imagenes(imagenes, modelo=None):
    """Convierte imágenes al formato que espera el modelo.

    Acepta una imagen (28, 28) o (28, 28, 1), o un lote (n, 28, 28) o (n, 28, 28, 1),
    con píxeles uint8 de 0 a 255 o float de 0 a 1. Si el modelo espera otra entrada
    (por ejemplo MobileNetV2 con 32 x 32 x 3), la convierte a RGB y la redimensiona.
    """
    original = np.asarray(imagenes)
    x = original.astype("float32")
    if x.shape in [(28, 28), (28, 28, 1)]:
        x = x.reshape(1, 28, 28, 1)
    elif x.ndim == 3 and x.shape[1:] == (28, 28):
        x = x[..., np.newaxis]
    elif not (x.ndim == 4 and x.shape[1:] == (28, 28, 1)):
        raise ValueError(
            f"Forma no válida {original.shape}: se espera (28, 28), (28, 28, 1) o un lote de esas imágenes."
        )
    if np.issubdtype(original.dtype, np.integer) or x.max() > 1.0:
        x = x / 255.0
    x = np.clip(x, 0.0, 1.0)

    if modelo is not None:
        alto, ancho, canales = modelo.input_shape[1:4]
        if canales == 3:
            x = tf.image.grayscale_to_rgb(tf.convert_to_tensor(x))
        if (alto, ancho) != (28, 28):
            x = tf.image.resize(x, [alto, ancho])
        x = np.asarray(x, dtype="float32")
    return x


def predecir_prenda(modelo, imagen, top=3):
    """Recibe UNA imagen de 28 x 28 y devuelve categoría, nombre de la prenda y probabilidad."""
    x = preparar_imagenes(imagen, modelo)
    if len(x) != 1:
        raise ValueError("predecir_prenda recibe una sola imagen; para varias usa predecir_lote.")
    inicio = time.perf_counter()
    probabilidades = np.asarray(modelo(x, training=False))[0]
    tiempo_ms = 1000 * (time.perf_counter() - inicio)
    orden = np.argsort(probabilidades)[::-1][:top]
    return {
        "categoria": int(orden[0]),
        "nombre": CATEGORIAS[int(orden[0])],
        "probabilidad": round(float(probabilidades[orden[0]]), 4),
        "top": [
            {"categoria": int(i), "nombre": CATEGORIAS[int(i)], "probabilidad": round(float(probabilidades[i]), 4)}
            for i in orden
        ],
        "tiempo_ms": round(tiempo_ms, 2),
    }


def predecir_lote(modelo, imagenes, tamano_lote=256):
    """Predice varias imágenes. Devuelve categorías, probabilidad de la categoría y matriz de probabilidades."""
    x = preparar_imagenes(imagenes, modelo)
    probabilidades = modelo.predict(x, batch_size=tamano_lote, verbose=0)
    return probabilidades.argmax(axis=1), probabilidades.max(axis=1), probabilidades


def medir_tiempo_prediccion(modelo, imagenes, repeticiones=100, calentamiento=10, tamano_lote=256):
    """Tiempo promedio de predicción.

    - Individual: una imagen por llamada, como la recibiría el robot. Se descartan las
      primeras llamadas de calentamiento.
    - En lote: todas las imágenes con predict y lotes de tamano_lote, dividido entre el número de imágenes.
    El tiempo individual incluye solo el modelo; la conversión de imágenes se hace antes.
    """
    x = preparar_imagenes(imagenes, modelo)
    for i in range(calentamiento):
        np.asarray(modelo(x[i % len(x)][np.newaxis], training=False))
    tiempos = []
    for i in range(repeticiones):
        imagen = x[i % len(x)][np.newaxis]
        inicio = time.perf_counter()
        np.asarray(modelo(imagen, training=False))
        tiempos.append(time.perf_counter() - inicio)
    modelo.predict(x[:tamano_lote], batch_size=tamano_lote, verbose=0)
    inicio = time.perf_counter()
    modelo.predict(x, batch_size=tamano_lote, verbose=0)
    total = time.perf_counter() - inicio
    ms = np.array(tiempos) * 1000
    return {
        "ms_por_imagen_individual": float(ms.mean()),
        "ms_desviacion_individual": float(ms.std()),
        "ms_por_imagen_en_lote": float(1000 * total / len(x)),
        "imagenes_por_segundo_en_lote": float(len(x) / total),
    }


def _principal():
    """Prueba en sesión limpia: carga el modelo y predice imágenes de validación sin variables previas."""
    parser = argparse.ArgumentParser(description="Prueba de inferencia de StyleBot en una sesión limpia")
    parser.add_argument("--modelo", default="modelos/stylebot_cnn_final.keras")
    parser.add_argument("--n", type=int, default=5, help="Número de imágenes de validación")
    args = parser.parse_args()

    from sklearn.model_selection import train_test_split

    inicio = time.perf_counter()
    modelo = cargar_modelo(args.modelo)
    print(f"Modelo cargado: {args.modelo} ({tamano_modelo_mb(args.modelo):.2f} MB) "
          f"en {time.perf_counter() - inicio:.2f} s")

    (X_base, y_base), _ = tf.keras.datasets.fashion_mnist.load_data()
    _, idx_val = train_test_split(np.arange(len(y_base), dtype=np.int64), test_size=0.20,
                                  random_state=42, stratify=y_base)
    for i in idx_val[: args.n]:
        resultado = predecir_prenda(modelo, X_base[i])  # imagen original uint8 de 0 a 255
        resultado["categoria_real"] = int(y_base[i])
        resultado["nombre_real"] = CATEGORIAS[int(y_base[i])]
        print(json.dumps(resultado, ensure_ascii=False))


if __name__ == "__main__":
    _principal()

In [ ]:
import importlib

sys.path.insert(0, str(Path.cwd()))
import inferencia_stylebot
importlib.reload(inferencia_stylebot)
from inferencia_stylebot import (
    CATEGORIAS, cargar_modelo, copia_para_inferencia, guardar_modelo, medir_tiempo_prediccion,
    predecir_lote, predecir_prenda, preparar_imagenes, tamano_modelo_mb,
)
print("Funciones cargadas desde", inferencia_stylebot.__file__)

## 4 Prueba provisional de las funciones (jueves)

Se carga la Arquitectura A y se predice una sola imagen de validación. También se comprueba que la función acepta la imagen en los tres formatos posibles y devuelve lo mismo.

In [ ]:
ruta_a = CARPETA_MODELOS / "arquitectura_a.keras"
modelo_prueba = cargar_modelo(ruta_a)

resultado = predecir_prenda(modelo_prueba, X_val_uint8[0])
print(json.dumps(resultado, ensure_ascii=False, indent=2))
print("Categoría real:", CATEGORIAS[y_val[0]])

for descripcion, entrada in [("uint8 (28, 28)", X_val_uint8[0]), ("float (28, 28, 1)", X_val[0]), ("float (28, 28)", X_val[0, :, :, 0])]:
    r = predecir_prenda(modelo_prueba, entrada)
    print(f"{descripcion:18} -> {r['nombre']} ({r['probabilidad']:.4f})")
    assert r["categoria"] == resultado["categoria"]

print(f"\nTamaño de arquitectura_a.keras: {tamano_modelo_mb(ruta_a):.2f} MB")
prueba_tiempo = medir_tiempo_prediccion(modelo_prueba, X_val[:200], repeticiones=30)
print("Tiempo provisional:", {k: round(v, 3) for k, v in prueba_tiempo.items()})

## 5 Guardar y volver a cargar la Arquitectura A (viernes)

1. Se carga `modelos/arquitectura_a.keras` (Arquitectura A sin aumento, guardada por Juan José).
2. Se vuelve a guardar con `guardar_modelo` y se carga de nuevo desde ese archivo.
3. Se comprueba que el modelo recargado da **exactamente** las mismas probabilidades.
4. Se predicen 10 imágenes de validación mostrando predicción, categoría real, probabilidad, tiempo y tamaño.

In [ ]:
modelo_a = cargar_modelo(ruta_a)
ruta_copia_a = guardar_modelo(modelo_a, SALIDA / "modelos" / "arquitectura_a_guardada_de_nuevo.keras")
modelo_a_recargado = cargar_modelo(ruta_copia_a)

prob_original = modelo_a.predict(X_val[:1000], verbose=0)
prob_recargado = modelo_a_recargado.predict(X_val[:1000], verbose=0)
diferencia = float(np.abs(prob_original - prob_recargado).max())
iguales = bool((prob_original.argmax(1) == prob_recargado.argmax(1)).all())
print(f"Diferencia máxima de probabilidades en 1,000 imágenes: {diferencia:.2e}")
print("Mismas categorías predichas:", iguales)
assert iguales and diferencia < 1e-5

tiempos_a = medir_tiempo_prediccion(modelo_a_recargado, X_val[:1000])
print(f"Tamaño original: {tamano_modelo_mb(ruta_a):.2f} MB | guardado de nuevo: {tamano_modelo_mb(ruta_copia_a):.2f} MB")
print(f"Tiempo promedio por imagen: {tiempos_a['ms_por_imagen_individual']:.2f} ms individual, "
      f"{tiempos_a['ms_por_imagen_en_lote']:.3f} ms en lote")

rng = np.random.default_rng(SEMILLA)
muestra = rng.choice(len(X_val), size=10, replace=False)
filas = []
for i in muestra:
    r = predecir_prenda(modelo_a_recargado, X_val_uint8[i])
    filas.append({
        "imagen_validacion": int(i), "prediccion": r["nombre"], "categoria_real": CATEGORIAS[y_val[i]],
        "probabilidad": r["probabilidad"], "correcta": r["categoria"] == int(y_val[i]), "tiempo_ms": r["tiempo_ms"],
    })
tabla_viernes = pd.DataFrame(filas)
tabla_viernes["tamano_modelo_mb"] = round(tamano_modelo_mb(ruta_copia_a), 2)
display(tabla_viernes)
tabla_viernes.to_csv(SALIDA / "resultados" / "inferencia_arquitectura_a_predicciones.csv", index=False)

fig, ejes = plt.subplots(2, 5, figsize=(13, 6))
for eje, (_, fila) in zip(ejes.ravel(), tabla_viernes.iterrows()):
    eje.imshow(X_val_uint8[fila["imagen_validacion"]], cmap="gray")
    eje.set_title(f"Pred: {fila['prediccion']} ({fila['probabilidad']:.2f})\nReal: {fila['categoria_real']}\n{fila['tiempo_ms']:.1f} ms",
                  fontsize=9, color="#1f7a3a" if fila["correcta"] else "#b02a2a")
    eje.axis("off")
fig.suptitle("Arquitectura A recargada desde .keras: 10 imágenes de validación")
fig.tight_layout()
fig.savefig(SALIDA / "resultados" / "inferencia_arquitectura_a_ejemplos.png", dpi=130, bbox_inches="tight")
plt.show()

## 6 Inferencia de las arquitecturas A, B y C (sábado)

Cada modelo se abre desde su archivo, como en una sesión nueva. Para cada uno se mide:

- **Parámetros** y **tamaño del archivo `.keras`**. El archivo de entrenamiento incluye el estado del optimizador Adam, que el robot no necesita.
- **Tamaño para inferencia**: el mismo modelo guardado sin optimizador con `copia_para_inferencia`. Es el que se compara entre modelos.
- **Tiempo de carga** del archivo.
- **Tiempo por imagen**: individual (una imagen por llamada, como en el robot) y en lote.
- **Accuracy y F1 macro en validación**, calculados aquí para confirmar que el modelo cargado es el entrenado. Se comparan con la tabla de Juan José.

Si algún archivo no está en `modelos/`, se reconstruye la arquitectura con el mismo código de `01_arquitecturas_cnn.ipynb`. Esa versión no tiene pesos entrenados, así que solo sirve para medir tamaño y tiempo, que no dependen de los valores de los pesos.

In [ ]:
from tensorflow.keras import layers, models


def crear_arquitectura_b():
    """Igual que en 01_arquitecturas_cnn.ipynb (Juan José), sin compilar."""
    modelo = models.Sequential(name="Arquitectura_B_Intermedia")
    modelo.add(layers.Input(shape=(28, 28, 1)))
    modelo.add(layers.Conv2D(32, (3, 3), padding="same", use_bias=False))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.Activation("relu"))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Dropout(0.25, seed=SEMILLA))
    modelo.add(layers.Conv2D(64, (3, 3), padding="same", use_bias=False))
    modelo.add(layers.BatchNormalization())
    modelo.add(layers.Activation("relu"))
    modelo.add(layers.MaxPooling2D((2, 2)))
    modelo.add(layers.Dropout(0.25, seed=SEMILLA))
    modelo.add(layers.Flatten())
    modelo.add(layers.Dense(128, activation="relu"))
    modelo.add(layers.Dropout(0.4, seed=SEMILLA))
    modelo.add(layers.Dense(10, activation="softmax"))
    return modelo


def crear_mobilenetv2():
    """Igual que en 01_arquitecturas_cnn.ipynb. weights=None: el tamaño y el tiempo son los mismos que con pesos de ImageNet."""
    base = tf.keras.applications.MobileNetV2(input_shape=(32, 32, 3), include_top=False, weights=None)
    entradas = layers.Input(shape=(32, 32, 3), name="entrada_rgb_32x32")
    x = layers.Rescaling(scale=2.0, offset=-1.0, name="escalado_mobilenet")(entradas)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3, seed=SEMILLA)(x)
    salidas = layers.Dense(10, activation="softmax")(x)
    return models.Model(inputs=entradas, outputs=salidas, name="MobileNetV2_TransferLearning")


tabla_equipo = pd.read_csv(RAIZ / "resultados" / "entrega2" / "comparacion_modelos_cnn_validacion.csv")
display(tabla_equipo)


def obtener_modelo(archivos, constructor=None):
    """Carga el primer archivo que exista; si no hay ninguno, reconstruye la arquitectura."""
    for nombre in archivos:
        ruta = CARPETA_MODELOS / nombre
        if ruta.exists():
            inicio = time.perf_counter()
            modelo = cargar_modelo(ruta)
            return modelo, ruta, time.perf_counter() - inicio, "archivo entrenado"
    if constructor is None:
        raise FileNotFoundError(f"No se encontró ninguno de {archivos}")
    return constructor(), None, None, "arquitectura reconstruida (sin pesos entrenados)"


def medir_modelo(nombre, nombre_equipo, modelo, ruta, tiempo_carga, origen):
    ruta_inferencia = copia_para_inferencia(modelo, SALIDA / "modelos" / f"{nombre.replace(' ', '_').lower()}_inferencia.keras")
    inicio = time.perf_counter()
    preparar_imagenes(X_val[:1000], modelo)
    ms_preparacion = 1000 * (time.perf_counter() - inicio) / 1000
    tiempos = medir_tiempo_prediccion(modelo, X_val[:1000])
    fila_equipo = tabla_equipo[tabla_equipo["Modelo"] == nombre_equipo]
    fila = {
        "modelo": nombre,
        "origen": origen,
        "parametros": modelo.count_params(),
        "parametros_tabla_equipo": int(fila_equipo["Parámetros"].iloc[0]) if len(fila_equipo) else None,
        "archivo_keras_mb": round(tamano_modelo_mb(ruta), 2) if ruta else None,
        "archivo_inferencia_mb": round(tamano_modelo_mb(ruta_inferencia), 2),
        "carga_s": round(tiempo_carga, 3) if tiempo_carga else None,
        "ms_preparacion_por_imagen": round(ms_preparacion, 4),
        "ms_por_imagen_individual": round(tiempos["ms_por_imagen_individual"], 3),
        "ms_por_imagen_en_lote": round(tiempos["ms_por_imagen_en_lote"], 4),
        "imagenes_por_segundo_en_lote": round(tiempos["imagenes_por_segundo_en_lote"]),
        "val_accuracy_calculada": None,
        "val_f1_macro_calculado": None,
        "val_accuracy_tabla_equipo": float(fila_equipo["Val Accuracy"].iloc[0]) if len(fila_equipo) else None,
        "val_f1_macro_tabla_equipo": float(fila_equipo["Val F1 Macro"].iloc[0]) if len(fila_equipo) else None,
    }
    if ruta is not None:
        predicciones, _, _ = predecir_lote(modelo, X_val)
        fila["val_accuracy_calculada"] = round(accuracy_score(y_val, predicciones), 4)
        fila["val_f1_macro_calculado"] = round(f1_score(y_val, predicciones, average="macro"), 4)
    return fila


filas_abc = []
for nombre, nombre_equipo, archivos, constructor in [
    ("Arquitectura A", "Arquitectura A (Sin aumento)", ["arquitectura_a.keras"], None),
    ("Arquitectura B", "Arquitectura B (Intermedia)", ["arquitectura_b.keras"], crear_arquitectura_b),
    ("Arquitectura C", "Arquitectura C (Profunda)", ["arquitectura_c.keras", "stylebot_cnn_final.keras"], None),
]:
    modelo, ruta, tiempo_carga, origen = obtener_modelo(archivos, constructor)
    print(f"{nombre}: {origen}{' -> ' + ruta.name if ruta else ''}")
    filas_abc.append(medir_modelo(nombre, nombre_equipo, modelo, ruta, tiempo_carga, origen))

tabla_abc = pd.DataFrame(filas_abc)
display(tabla_abc)
tabla_abc.to_csv(SALIDA / "resultados" / "inferencia_abc.csv", index=False)

for fila in filas_abc:
    if fila["parametros_tabla_equipo"] is not None:
        assert fila["parametros"] == fila["parametros_tabla_equipo"], f"Parámetros distintos en {fila['modelo']}"
print("Parámetros iguales a los de la tabla de Juan José en los tres modelos.")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
nombres = tabla_abc["modelo"]
colores = ["#7aa6c2", "#326b9b", "#1d3f5e"]
ejes[0].bar(nombres, tabla_abc["archivo_inferencia_mb"], color=colores)
ejes[0].set_title("Tamaño para inferencia (MB)")
ejes[1].bar(nombres, tabla_abc["ms_por_imagen_individual"], color=colores)
ejes[1].set_title("Tiempo por imagen individual en CPU (ms)")
ejes[2].bar(nombres, tabla_abc["val_accuracy_calculada"].fillna(tabla_abc["val_accuracy_tabla_equipo"]), color=colores)
ejes[2].set_ylim(0.85, 0.95)
ejes[2].set_title("Accuracy en validación")
for eje in ejes:
    eje.tick_params(axis="x", labelsize=9)
fig.tight_layout()
fig.savefig(SALIDA / "resultados" / "inferencia_abc.png", dpi=130, bbox_inches="tight")
plt.show()

### ¿Cuál es más práctica para el robot?

*Se completa con los resultados de la ejecución.*

## 7 CNN propia contra transfer learning con MobileNetV2 (domingo)

Se compara la mejor CNN propia elegida con validación (Arquitectura C, `stylebot_cnn_final.keras`) contra MobileNetV2. MobileNetV2 necesita convertir cada imagen a 3 canales y redimensionarla a 32 x 32; ese costo aparece en la columna de preparación.

Si `modelos/` no contiene el MobileNetV2 entrenado, se reconstruye con la misma arquitectura para medir tamaño y tiempo, y su accuracy se toma de la tabla de validación de Juan José.

In [ ]:
modelo_c, ruta_c, carga_c, origen_c = obtener_modelo(["stylebot_cnn_final.keras"])
modelo_mn, ruta_mn, carga_mn, origen_mn = obtener_modelo(["mobilenetv2.keras", "mobilenetv2_transfer.keras"], crear_mobilenetv2)
print("MobileNetV2:", origen_mn)

filas_tl = [
    medir_modelo("CNN propia C (final)", "Arquitectura C (Profunda)", modelo_c, ruta_c, carga_c, origen_c),
    medir_modelo("MobileNetV2", "MobileNetV2 (Transfer Learning)", modelo_mn, ruta_mn, carga_mn, origen_mn),
]
tabla_tl = pd.DataFrame(filas_tl)
display(tabla_tl)
tabla_tl.to_csv(SALIDA / "resultados" / "inferencia_cnn_vs_mobilenetv2.csv", index=False)
assert filas_tl[1]["parametros"] == filas_tl[1]["parametros_tabla_equipo"], "MobileNetV2 no coincide con la de Juan José"

c, m = filas_tl
print(f"MobileNetV2 tiene {m['parametros'] / c['parametros']:.1f} veces los parámetros de la CNN C, "
      f"pesa {m['archivo_inferencia_mb'] / c['archivo_inferencia_mb']:.1f} veces más "
      f"y tarda {m['ms_por_imagen_individual'] / c['ms_por_imagen_individual']:.1f} veces más por imagen.")

### ¿Cuál conviene para la simulación?

*Se completa con los resultados de la ejecución.*

## 8 Modelo final en una sesión limpia y formato para la Entrega 3 (lunes)

Para demostrar que el modelo final no depende de variables de este notebook, se ejecuta `inferencia_stylebot.py` en **otro proceso de Python**, que empieza vacío: carga `stylebot_cnn_final.keras`, carga los datos y predice imágenes de validación originales (`uint8`, de 0 a 255).

In [ ]:
comando = [sys.executable, str(Path("inferencia_stylebot.py").resolve()),
           "--modelo", str(CARPETA_MODELOS / "stylebot_cnn_final.keras"), "--n", "8"]
proceso = subprocess.run(comando, capture_output=True, text=True, cwd=RAIZ)
if proceso.returncode != 0:
    print(proceso.stderr[-3000:])
    raise RuntimeError("La prueba en sesión limpia falló")

lineas = proceso.stdout.strip().splitlines()
print(next(l for l in lineas if l.startswith("Modelo cargado")))
resultados_limpios = [json.loads(l) for l in lineas if l.startswith("{")]
assert len(resultados_limpios) == 8
tabla_limpia = pd.DataFrame([{
    "prediccion": r["nombre"], "categoria_real": r["nombre_real"], "probabilidad": r["probabilidad"],
    "correcta": r["categoria"] == r["categoria_real"], "tiempo_ms": r["tiempo_ms"],
    "segunda_opcion": f"{r['top'][1]['nombre']} ({r['top'][1]['probabilidad']:.3f})",
} for r in resultados_limpios])
display(tabla_limpia)
tabla_limpia.to_csv(SALIDA / "resultados" / "inferencia_modelo_final_sesion_limpia.csv", index=False)
print("Ejemplo de salida completa:")
print(json.dumps(resultados_limpios[0], ensure_ascii=False, indent=2))

In [ ]:
modelo_final = cargar_modelo(CARPETA_MODELOS / "stylebot_cnn_final.keras")
rng = np.random.default_rng(SEMILLA + 1)
muestra_final = rng.choice(len(X_val), size=12, replace=False)

fig, ejes = plt.subplots(2, 6, figsize=(15, 6))
for eje, i in zip(ejes.ravel(), muestra_final):
    r = predecir_prenda(modelo_final, X_val_uint8[i])
    correcta = r["categoria"] == int(y_val[i])
    eje.imshow(X_val_uint8[i], cmap="gray")
    eje.set_title(f"Pred: {r['nombre']} ({r['probabilidad']:.2f})\nReal: {CATEGORIAS[y_val[i]]}", fontsize=9,
                  color="#1f7a3a" if correcta else "#b02a2a")
    eje.axis("off")
fig.suptitle("Modelo final (stylebot_cnn_final.keras): 12 imágenes de validación")
fig.tight_layout()
fig.savefig(SALIDA / "resultados" / "inferencia_modelo_final_ejemplos.png", dpi=130, bbox_inches="tight")
plt.show()

### Formato de entrada y salida para la Entrega 3

**Modelo:** `modelos/stylebot_cnn_final.keras` (Arquitectura C), cargado con `cargar_modelo`.

**Entrada:** una imagen de la prenda.

| Aspecto | Formato |
|---|---|
| Tamaño | 28 x 28 píxeles |
| Color | Escala de grises, un canal |
| Forma aceptada | `(28, 28)` o `(28, 28, 1)`; para varias imágenes, `(n, 28, 28)` o `(n, 28, 28, 1)` |
| Valores | `uint8` de 0 a 255 o `float32` de 0 a 1 (la función normaliza sola) |
| Estilo | Prenda clara centrada sobre fondo oscuro, como Fashion-MNIST |

**Salida de `predecir_prenda`:** un diccionario que se puede convertir a JSON.

| Campo | Tipo | Ejemplo |
|---|---|---|
| `categoria` | entero 0-9 | `9` |
| `nombre` | texto | `"Botín"` |
| `probabilidad` | número de 0 a 1 | `0.9987` |
| `top` | lista de 3 opciones con categoría, nombre y probabilidad | segunda y tercera opción |
| `tiempo_ms` | milisegundos de la predicción | `35.2` |

Si la imagen de la cámara de la simulación no tiene este formato, la Entrega 3 debe recortarla, pasarla a escala de grises y reducirla a 28 x 28 antes de llamar a `predecir_prenda`. Cuando `probabilidad` sea baja, el robot puede usar `top` para pedir confirmación en lugar de actuar.

## 9 Descargar resultados

Crea un ZIP con las tablas y figuras, que van a `resultados/entrega2/inferencia/` del repositorio. Los modelos copiados en `salidas_03_inferencia/modelos/` no se incluyen, porque son copias de prueba.

In [ ]:
import zipfile

ruta_zip = Path("StyleBot_03_inferencia_resultados.zip")
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as comprimido:
    for ruta in sorted((SALIDA / "resultados").glob("*")):
        comprimido.write(ruta, ruta.name)
print("Archivos:", [p.name for p in sorted((SALIDA / "resultados").glob("*"))])

try:
    from google.colab import files
except ImportError:
    print("Fuera de Colab: el ZIP está en", ruta_zip.resolve())
else:
    files.download(str(ruta_zip))

## Fuentes

- Keras. Guardar y cargar modelos (`model.save`, `keras.models.load_model`, formato `.keras`). https://keras.io/guides/serialization_and_saving/
- Keras. `clone_model`. https://keras.io/api/models/model_saving_apis/model_cloning/
- TensorFlow. `tf.config.set_visible_devices`. https://www.tensorflow.org/api_docs/python/tf/config/set_visible_devices
- TensorFlow. `tf.image.grayscale_to_rgb` y `tf.image.resize`. https://www.tensorflow.org/api_docs/python/tf/image
- Python. `time.perf_counter`. https://docs.python.org/3/library/time.html#time.perf_counter
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A. y Chen, L.-C. (2018). MobileNetV2: Inverted Residuals and Linear Bottlenecks. CVPR. https://doi.org/10.1109/CVPR.2018.00474
- Equipo StyleBot. `00_datos_cnn.ipynb` (división), `01_arquitecturas_cnn.ipynb` (arquitecturas B y MobileNetV2, tabla de validación) y `modelos/*.keras`.

El código se elaboró con asistencia de IA y fue revisado por el responsable, según las reglas del curso.